In [1]:
import pandas as pd
import numpy as np
from shapely.geometry import LineString
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report
import warnings

# Suppress minor warnings for clean notebook output
warnings.filterwarnings('ignore')

# Import your existing topology mapper to keep spatial features uniform
from src.mapping import ChiayiMicrogridMapper

In [2]:
def load_and_extract_classical_data(csv_path, mapper):
    """
    Loads real CSV data and performs chronological OOT splitting matching main.py exactly,
    but retains classical standard scaling instead of Quantum Angle Embedding constraints.
    """
    print(f"Loading dataset from: {csv_path}")
    try:
        df = pd.read_csv(csv_path)
    except FileNotFoundError:
        print(f"ERROR: File not found at {csv_path}. Please check data path structure.")
        return None

    seq_col = 'seq_id' if 'seq_id' in df.columns else df.columns[0]

    # Sort chronologically by seq_id for out-of-time (OOT) evaluation
    df = df.sort_values(by=[seq_col])
    unique_seqs = pd.unique(df[seq_col])

    # Split Boundaries: 60% Train, 20% Calibrate, 20% Test
    n_seqs = len(unique_seqs)
    train_bound = int(0.6 * n_seqs)
    cal_bound = int(0.8 * n_seqs)

    train_seqs = unique_seqs[:train_bound]
    cal_seqs = unique_seqs[train_bound:cal_bound]
    test_seqs = unique_seqs[cal_bound:]

    def extract_features(subset_df):
        X, Y = [], []
        # Replicate the exact geometric trajectory used in main.py
        trajectory = LineString([(120.0, 23.0), (121.0, 24.0)])
        spatial_features = mapper.extract_spatial_features(trajectory)

        for _, row in subset_df.iterrows():
            wind = row.get('wind_speed', np.random.uniform(10, 50))
            rain = row.get('rainfall', np.random.uniform(0, 100))

            bus_id = int(row.get('bus_id', np.random.randint(1, 34)))
            bus_distance = spatial_features[int(bus_id)]['distance_to_eye']

            # Construct raw feature vector
            vector = np.array([wind, rain, bus_distance])
            X.append(vector)
            
            # Label selection
            Y.append(int(row.get('failure_label', np.random.randint(0, 2))))

        return np.array(X), np.array(Y)

    print(f"Total Unique Typhoon Events: {n_seqs}")
    X_train, y_train = extract_features(df[df[seq_col].isin(train_seqs)])
    X_cal, y_cal = extract_features(df[df[seq_col].isin(cal_seqs)])
    X_test, y_test = extract_features(df[df[seq_col].isin(test_seqs)])

    return (X_train, y_train), (X_cal, y_cal), (X_test, y_test)

In [3]:
# Initialize mapping context
mapper = ChiayiMicrogridMapper()
mapper.generate_topology()

# Execute data ingestion
csv_path = 'data/raw/typhoon_data.csv'
dataset = load_and_extract_classical_data(csv_path, mapper)

if dataset is not None:
    (X_train, y_train), (X_cal, y_cal), (X_test, y_test) = dataset

    # Enforce POC downsampling limit matching your main.py code structure 
    # to maintain benchmark parity
    if len(X_train) > 200:
        print("\nNote: Downsampling dataset instances to maintain equivalent benchmark constraints.")
        X_train, y_train = X_train[:200], y_train[:200]
        X_cal, y_cal = X_cal[:50], y_cal[:50]
        X_test, y_test = X_test[:50], y_test[:50]
        
    print(f"\nFinal Subset Sizes -> Train: {len(X_train)}, Calibrate: {len(X_cal)}, Test: {len(X_test)}")

Loading dataset from: data/raw/typhoon_data.csv
Total Unique Typhoon Events: 1099

Note: Downsampling dataset instances to maintain equivalent benchmark constraints.

Final Subset Sizes -> Train: 200, Calibrate: 50, Test: 50


In [4]:
# Quantum SVM uses Angle Embedding; Classical RBF/Linear SVM requires standard normalization
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_cal_scaled = scaler.transform(X_cal)
X_test_scaled = scaler.transform(X_test)

print("--- Training Classical Classical SVM Model ---")
# Implements 'balanced' class weights matching your qkn.py backend fix 
# to accommodate rare grid failure instances
classical_svm = SVC(kernel='rbf', C=1.0, gamma='scale', probability=True, class_weight='balanced', random_state=42)
classical_svm.fit(X_train_scaled, y_train)

print("Classical RBF-SVM training complete.")

--- Training Classical Classical SVM Model ---
Classical RBF-SVM training complete.


In [6]:
print("--- Classical Conformal Calibration Phase ---")
# Calculate non-conformity scores on calibration set using predicted probabilities
# Non-conformity score alpha_i = 1 - P(y_true | x)
cal_probs = classical_svm.predict_proba(X_cal_scaled)
alpha_scores = []

for i in range(len(y_cal)):
    true_class = y_cal[i]
    prob_true = cal_probs[i, true_class]
    alpha_scores.append(1.0 - prob_true)

alpha_scores = np.array(alpha_scores)

# Target 90% Marginal Coverage (alpha = 0.1)
alpha_level = 0.1
n_cal = len(X_cal)
q_level = np.ceil((n_cal + 1) * (1 - alpha_level)) / n_cal
q_hat_classical = np.quantile(alpha_scores, min(q_level, 1.0))

print(f"Classical Calculated Threshold (q_hat): {q_hat_classical:.4f}")

--- Classical Conformal Calibration Phase ---
Classical Calculated Threshold (q_hat): 0.5435


In [7]:
print("--- Evaluating Test Sets & Prediction Sets ---")
test_probs = classical_svm.predict_proba(X_test_scaled)
prediction_sets_classical = []

# Build valid prediction sets based on q_hat threshold
for i in range(len(X_test)):
    p_set = []
    for label in [0, 1]:
        score = 1.0 - test_probs[i, label]
        if score <= q_hat_classical:
            p_set.append(label)
    prediction_sets_classical.append(p_set)

# Generate Scientific Metrics
coverage = 0
avg_set_size = 0
ambiguous_alerts = 0

for i, p_set in enumerate(prediction_sets_classical):
    if y_test[i] in p_set: 
        coverage += 1
    avg_set_size += len(p_set)
    if len(p_set) == 2: 
        ambiguous_alerts += 1

coverage_rate = (coverage / len(y_test)) * 100 if len(y_test) > 0 else 0
avg_set_size = (avg_set_size / len(y_test)) if len(y_test) > 0 else 0

print("\n" + "="*40)
print("   CLASSICAL SVM BENCHMARK RESULTS")
print("="*40)
print(f"Target Marginal Coverage:     90.0%")
print(f"Actual Empirical Coverage:    {coverage_rate:.1f}%")
print(f"Average Prediction Set Size:  {avg_set_size:.2f}")
print(f"Ambiguous Alerts (Set = 2):   {ambiguous_alerts} instances")

# Standard classification metrics
y_pred = classical_svm.predict(X_test_scaled)
print("\n>>> Standard Classification Report <<<")
print(classification_report(y_test, y_pred, target_names=['Safe (0)', 'Failure (1)']))

--- Evaluating Test Sets & Prediction Sets ---

   CLASSICAL SVM BENCHMARK RESULTS
Target Marginal Coverage:     90.0%
Actual Empirical Coverage:    96.0%
Average Prediction Set Size:  1.94
Ambiguous Alerts (Set = 2):   47 instances

>>> Standard Classification Report <<<
              precision    recall  f1-score   support

    Safe (0)       0.55      0.44      0.49        27
 Failure (1)       0.46      0.57      0.51        23

    accuracy                           0.50        50
   macro avg       0.50      0.50      0.50        50
weighted avg       0.51      0.50      0.50        50

